# eCREST JSON to VAST Skeleton Conversion

**Purpose:** Converts eCREST `.json` reconstruction files into VAST-compatible skeleton format
for import into VASTlite (Harvard Lichtman Lab). This notebook serves two pipeline roles:

**Inputs:**
- `EM_data_published/reconstructions_published/` — eCREST `.json` reconstruction files

**Outputs:**
- VAST skeleton files — input to VAST for segment agglomeration and export 

**Used by:** VAST for processing of segmentation layers 

**Launch requirement:** Run `jupyter lab` from `efish_em/Notebooks_Jupyter/`

# setup

In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
from tqdm import tqdm

from efish_em import AnalysisCode as efish #only works after the efish_em package has been installed as per README

In [4]:
DATA_ROOT = Path.cwd().parent.parent / 'EM_data_published'
dirpath = DATA_ROOT / 'reconstructions_published'
vx_sizes = [16, 16, 30]

# get bigquery table

In [ ]:
# location query run on all base segments in published reconstructions and saved as .parquet and .csv (so can load as either)
seg_bigquery_df = pd.read_parquet(DATA_ROOT / 'base-segs_query_published.parquet')

# file paths and loads


In [ ]:
DATA_ROOT = Path.cwd().parent.parent / 'EM_data_published'
df_type = pd.read_csv(DATA_ROOT / 'data_processed_published/df_type_auto_typed.csv')

In [ ]:
# VAST skeleton CSVs are written here. Default is a local 'outputs/vast_skeletons/'
# directory relative to this notebook; override as needed.
path2skeletondir = Path.cwd() / 'outputs' / 'vast_skeletons'
path2skeletondir.mkdir(parents=True, exist_ok=True)


# json from neuroglancer for 16nm VAST

In [14]:
cell_filepaths = efish.get_cell_filepaths(dirpath)  # gets filepaths for all cells in dirpath


In [ ]:
skeleton_folder = ''

with tqdm(total=len(cell_filepaths.keys())) as pbar:
        
    for k,v in cell_filepaths.items():
        pbar.update(1)
    
        with open(v) as f:
            data = json.load(f)
    
        # find the index of the segmentation layer
        seg_layer_idx = next(i for i, layer in enumerate(data['layers']) if layer['type'] == 'segmentation')
        
        base_seg_list = data['layers'][seg_layer_idx]['segments']
        # base_seg_str = ', '.join(str(x) for x in base_seg_list)
        
        print(f'working on {k}')

        df = seg_bigquery_df.query("seg_id in @base_seg_list")
        df = df.drop_duplicates(subset=['seg_id'])

        # saveing information
        timestr = time.strftime("%Y%m%d-%H%M%S")
        savename_and_path = path2skeletondir / skeleton_folder / f'{k}_skeleton_{timestr}.csv';
        df.to_csv(savename_and_path, index=False)

print("done!")


# json from eCREST for 16nm VAST

In [14]:
cell_filepaths = efish.get_cell_filepaths(dirpath)  # gets filepaths for all cells in dirpath


In [17]:

cell_list = [] # anchor segment IDs of recosntructions want skeleton for

## for only segments in a given region of each cell

In [18]:
skeleton_folder = 'subvolume_apical'

dtype='apical dendrite'
for ind,cell_id in enumerate(cell_list):
    data = efish.load_ecrest_celldata(cell_filepaths[str(cell_id)]) #crest.cell_data

    base_seg_list = [bs for bs in data['base_segments'][dtype]] # if only need segments from one cell structure 
    # [bs for dtype in data['base_segments'].keys() for bs in data['base_segments'][dtype]] # if need all segments
    # sum([i for i in data.get('base_segments').values()],[]); #
    
    print(f'working on {k}')

    df = seg_bigquery_df.query("seg_id in @base_seg_list")
    df = df.drop_duplicates(subset=['seg_id'])
    # df = df[df['y']<15000]
    ctype = df_type[df_type['id'].isin([cell_id])]['cell_type'].values[0]
    df.to_csv(path2skeletondir / skeleton_folder / f'{str(cell_id)}_{ctype}_ad.csv', index=False)

print("done!")

done!


## for all segments

In [19]:
cell_filepaths = efish.get_cell_filepaths(dirpath) # gets filepaths for all cells in a directory

In [25]:
cell_list = []

In [26]:
skeleton_folder = ''
for k in cell_list:
    v = cell_filepaths[k]
    data = efish.load_ecrest_celldata(v)#crest.cell_data
    ctype = df_type[df_type['id'].isin([int(k)])]['cell_type'].values[0]
    
    base_seg_list = [bs for dtype in data['base_segments'].keys() for bs in data['base_segments'][dtype]]  
    # base_seg_str = ', '.join(str(x) for x in base_seg_list)

    # saveing information
    savename_and_path = path2skeletondir  / skeleton_folder / f"{ctype}_{k}_skeleton.csv";

    if not os.path.exists(savename_and_path):
           
        df = seg_bigquery_df.query("seg_id in @base_seg_list")
        df = df.drop_duplicates(subset=['seg_id'])
        df.to_csv(path2skeletondir / skeleton_folder / f'{str(cell_id)}_{ctype}_all.csv', index=False)
print("done!")


done!
